In [1]:
import os
import sys
sys.path.append(r"C:\Users\Admin\Desktop\master_thesis\volume_prediction_fip")

import pandas as pd
from multiview_instance_epipole import multiview_graph
import json
import time

In [5]:
df = pd.read_csv(r"F:\FIP-data\csv\precomputed_old.csv")


In [6]:
with open(r"assets/poses.json") as f:
        poses_conf = json.load(f)

import numpy as np
def converter(obj):
        if isinstance(obj, np.int64):
                return int(obj)
        elif isinstance(obj, np.float32):
                return float(obj)

In [7]:
#mapping = {'cam_01.png': (0.28249773408636614, 9329.312471639554), 'cam_02.png': (0.3461241013576846, 9172.521471547236), 'cam_03.png': (0.2756008869149703, 9378.413691671205), 'cam_04.png': (0.40251108778355366, 9086.1502509331), 'cam_05.png': (0.25598905985567444, 9422.36970812002), 'cam_06.png': (0.2468143761085707, 9443.0805774625), 'cam_07.png': (0.1938961703501688, 9576.548003795544), 'cam_08.png': (0.06892766394265483, 9905.443103747255), 'cam_09.png': (0.20812736830548184, 9522.243724242306), 'cam_10.png': (0.21604982054565158, 9530.145510981583), 'cam_11.png': (0.25051134525389174, 9449.877630082683), 'cam_12.png': (0.20691745022331945, 9557.712715474687)}
#mappingmedian = {'cam_01.png': 10084.20741872331, 'cam_02.png': 10093.752921133455, 'cam_03.png': 10111.069353439112, 'cam_04.png': 10152.30522079368, 'cam_05.png': 10107.04373510269, 'cam_06.png': 10105.907137549271, 'cam_07.png': 10097.411479967595, 'cam_08.png': 10105.804682083586, 'cam_09.png': 10078.570105426234, 'cam_10.png': 10113.113802965998, 'cam_11.png': 10119.562465615089, 'cam_12.png': 10111.155510720793}

#mapping = {'cam_01.png': (0.17673331477901844, 9530.035254373397), 'cam_02.png': (0.17331549264743165, 9548.864223309742), 'cam_03.png': (0.19916796390992236, 9494.175132345183), 'cam_04.png': (0.17149694106928104, 9596.678688906339), 'cam_05.png': (0.24924534577586174, 9353.458675560674), 'cam_06.png': (0.22014159591312138, 9435.167727120324), 'cam_07.png': (0.251777615636003, 9347.220352681334), 'cam_08.png': (0.27099496141011836, 9292.946995161974), 'cam_09.png': (0.27684184329891653, 9263.55753990981), 'cam_10.png': (0.26332516569589537, 9324.141285825848), 'cam_11.png': (0.17323378607016485, 9582.929318933833), 'cam_12.png': (0.22748550981767562, 9427.521382588078)}
#mappingmedian = {'cam_01.png': 9988.005284587069, 'cam_02.png': 10000.154780064087, 'cam_03.png': 10011.983258332093, 'cam_04.png': 10040.549202144306, 'cam_05.png': 10006.760056020104, 'cam_06.png': 10010.456625782452, 'cam_07.png': 10009.562928577994, 'cam_08.png': 10005.420284922133, 'cam_09.png': 9996.683941619402, 'cam_10.png': 10013.474703016305, 'cam_11.png': 10034.323871629393, 'cam_12.png': 10023.233323789655}


for idx, row in df.iterrows():
    if r"2024_07_18_14_08_Lot3" in  row["image_dir"]:
        d = json.loads(row["spikes"])
        conv = {}
        for img, boxes in d.items():
            n = {}
            for i, (_, v) in enumerate(boxes):
                n[i] = v
            conv[img] = n

        # Adapt focal length with radar
        last = os.path.split(row["image_dir"])[1]
        with open(os.path.join(row["image_dir"], last.split("_")[0] + "_" + last.split("_")[1] + "_rig.json")) as f:
            radar = json.load(f)["plant_distance"]
        if radar is None:
            print(f"No radar for {row['image_dir']}")
            for k in poses_conf:
                poses_conf[k]["intrinsics"]["focal_length"] = mappingmedian[k]
        else:
            for k in poses_conf:
                alpha, offset = mapping[k]
                poses_conf[k]["intrinsics"]["focal_length"] = radar * alpha + offset
        
        graph, idx_to_instance = multiview_graph.build_epipolar_graph_opt(poses_conf, conv)
        graph = graph.cpu().numpy()
        labels = multiview_graph.lp_cluster_torch(graph).numpy()
        
        labelcount = {}
        for v in labels:
            labelcount.setdefault(v, 0)
            labelcount[v] += 1

        # Dict mapping from image name to id, bounding box. Unsure bounding boxes (less than min_view predicted views get label -1)
        combined_results = {} 
        for i in range(len(idx_to_instance)):
            image_name, id = idx_to_instance[i]
            box = conv[image_name][id]
            combined_results.setdefault(image_name, [])
            label = labels[i]
            combined_results[image_name].append((label, box))

        df.at[idx, "spikes"] = json.dumps(combined_results, default=converter)


No radar for F:\FIP-data\images\2024\WW036\debayered\2024_07_18_14_08_Lot3\FPWW0360254_FIP2_20240718_134638
No radar for F:\FIP-data\images\2024\WW036\debayered\2024_07_18_14_08_Lot3\FPWW0360313_FIP2_20240718_135307


In [8]:
df.to_csv(r"F:\FIP-data\csv\precomputed.csv")